In [ ]:
#OLINK BIOMARKER DATA PREP

"""
This script prepares the olink data for correlation with the cognitive z-scores.
It includes calculation of eGFR from creatinine, age and sex.
It then adds the biomarker data and the confounders to the cognitive scores for correlation
"""

import pandas as pd
import numpy as np
import os

#Set output direction
output = 'data'
os.makedirs(output, exist_ok=True)

#import data
cog = pd.read_csv('cv_results_all_tests_merged_with_groups.csv')
olink = pd.read_csv('olink_olink_instance_0.csv')
conf = pd.read_csv('olink_confounds.csv')
conf[:5]

In [ ]:
conf[:5]

In [ ]:
#PREPARE CONFOUNDS DATA

"""
This script transforms serum creatinine values to estimated Glomerular Filtration Rate (eGFR)
using the CKD-EPI 2021 equation (race-free version).

Handles sex coded as 'Male'/'Female' strings and converts to numeric (0/1).
Handles creatinine in μmol/L (converts to mg/dL).

Input: creatinine (μmol/L), age (years), sex ('Male' or 'Female')
Output: eGFR (mL/min/1.73m²)
"""

#Renames column headings
conf = conf.rename(columns = {
    'p21022': 'age',
    'p31': 'sex',
    'p21001_i0': 'BMI',
    'p30700_i0': 'creatinine'})

#Recode sex from male/female to 0/1
conf['sex'] = conf['sex'].map({
    'Male': '1',
    'Female': '0'})

#convert creatinine from umol/L to mg/dL
conf['creatinine'] = conf['creatinine']/88.42
conf[:5]

In [ ]:
#CALCULATE EGFR

"""
This block Calculates eGFR using the CKD-EPI 2021 equation.
it first defines a function, eGFR_calc for the calculation steps, then applies it to the data
The function first defines the the mathematical terms, then combines them into a formula.

Equation: eGFR = 142 × min(Scr/κ, 1)^α × max(Scr/κ, 1)^-1.200 × 0.9938^Age × 1.012 [if female]

Where:
- κ = 0.7 (females) or 0.9 (males)
- α = -0.241 (females) or -0.302 (males)
- Scr must be in mg/dL

Handles sex coded as 'Male'/'Female' strings and converts to numeric (0/1).
Handles creatinine in μmol/L (converts to mg/dL by dividing by 88.4).

Input: creatinine (mg/dL), age (years), sex ('Male' or 'Female')
Output: eGFR (mL/min/1.73m²)
"""

def eGFR_calc(creatinine, age, sex):
    #set constants using np.where(condition, value_if_true, value_if_false)
    kappa = np.where(sex == 1, 0.7, 0.9)  
    alpha = np.where(sex == 1, -0.241, -.302)
    scr_kappa = creatinine/kappa #standardise creatinine
    min_term = np.minimum(scr_kappa, 1) ** alpha
    max_term = np.maximum(scr_kappa, 1) ** (-1.200)
    age_term = 0.9938 ** age
    sex_coef = np.where(sex == 1, 1.012, 1)
    eGFR = 142 * min_term * max_term * age_term * sex_coef
    return eGFR

conf['eGFR'] = eGFR_calc(
    creatinine = conf['creatinine'],
    age = conf['age'],
    sex = conf['sex'])

conf[:5]
conf.to_csv('biomarker_confounders.csv', index=False)

In [ ]:
#MERGE THE CONFOUNDER DATA TO OLINK DATA

"""
This blocks merges only the needed columns of confounders for the olink datat
"""

olink = olink.rename(columns={'olink_instance_0$eid': 'eid'})

biomarkers = pd.merge(conf, olink[['eid','gfap', 'nefl']], on='eid', how='inner')
biomarkers = biomarkers.sort_values(by='eid')
biomarkers = biomarkers.drop(columns=['creatinine'])
biomarkers = biomarkers.dropna()
biomarkers.to_csv('data/olink_prep.csv', index=False)
biomarkers[:5]


biomarkers = pd.read_csv('data/olink_prep.csv')

merge = pd.merge(cog, biomarkers[['eid', 'BMI', 'eGFR', 'gfap', 'nefl']], on='eid', how='inner')
merge.to_csv('cog_biomarker_merged.csv', index=False)
display(merge[:5])

len(merge)